[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensioai/blog/blob/master/028_pytorch_nn/pytorch_nn.ipynb)

# Pytorch - Redes Neuronales

En el post [anterior](https://sensioai.com/blog/027_pytorch_intro) hicimos una introducción al framework de `redes neuronales` `Pytorch`. Hablamos de sus tres elementos fundamentales: el objeto `tensor` (similar al `array` de `NumPy`) `autograd` (que nos permite calcular derivadas de manera automáticas) y el soporte GPU. En este post vamos a entrar en detalle en la  funcionalidad que nos ofrece la librería para diseñar redes neuronales de manera flexible.

In [1]:
import torch

## Modelos secuenciales

La forma más sencilla de definir una `red neuronal` en `Pytorch` es utilizando la clase `Sequentail`. Esta clase nos permite definir una secuencia de capas, que se aplicarán de manera secuencial (las salidas de una capa serán la entrada de la siguiente). Ésto ya lo conocemos de posts anteriores, ya que es la forma ideal de definir un `Perceptrón Multicapa`.

In [2]:
D_in, H, D_out = 784, 100, 10

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
)

# D_in, H1, H2, D_out = 784, 100, 50, 10
# model = torch.nn.Sequential(
#     torch.nn.Linear(D_in, H1),
#     torch.nn.ReLU(),
#     torch.nn.Linear(H1, H2),
#     torch.nn.ReLU(),
#     torch.nn.Linear(H2, D_out),
# )


El modelo anterior es un `MLP` con 784 entradas, 100 neuronas en la capa oculta y 10 salidas. Podemos usar este modelo para hacer un clasificador de imágenes con el dataset MNIST. Pero primero, vamos a ver como podemos calcular las salidas del modelo a partir de unas entradas de ejemplo.

In [3]:
outputs = model(torch.randn(600, 784))
outputs.shape

torch.Size([600, 10])

In [4]:
print(outputs[0][:])

tensor([-0.1547, -0.0589,  0.3917,  0.0078, -0.1044,  0.0331,  0.4700, -0.4089,
         0.0492,  0.0083], grad_fn=<SliceBackward0>)


Como puedes ver, simplemente le pasamos los inputs al modelo (llamándolo como una función). En este caso, usamos un tensor con 64 vectores de 784 valores. Es importante remarcar que los modelos de `Pytorch` (por lo general) siempre esperan que la primera dimensión sea la dimensión *batch*. Si queremos entrenar esta red en una GPU, es tan sencillo como

In [5]:
model

Sequential(
  (0): Linear(in_features=784, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=10, bias=True)
)

In [6]:
#model.to("cuda")

Vamos a ver ahora como entrenar este modelo con el dataset MNIST.

In [8]:
import pandas as pd
from pathlib import Path

# localizar el dataset junto a la notebook o desde la raiz del workspace
#rutas = [Path("secondary_data.csv"), Path("avance") / "secondary_data.csv"]
#ruta = next((ruta for ruta in rutas if ruta.exists()), None)
#if ruta is None:
#    raise FileNotFoundError("No se encontro secondary_data.csv")

df = pd.read_csv("secondary_data.csv", sep=";")
print(f"Dataset: {df.shape[0]} | Filas: {len(df)} | Columnas: {len(df.columns)}")
df.head()

Dataset: 61069 | Filas: 61069 | Columnas: 21


,class,cap-diameter,cap-shape,cap-surface,cap-color,does-bruise-or-bleed,gill-attachment,gill-spacing,gill-color,stem-height,...,stem-root,stem-surface,stem-color,veil-type,veil-color,has-ring,ring-type,spore-print-color,habitat,season
0,p,15.26,x,g,o,f,e,NaN,w,16.95,...,s,y,w,u,w,t,g,NaN,d,w
1,p,16.60,x,g,o,f,e,NaN,w,17.99,...,s,y,w,u,w,t,g,NaN,d,u
2,p,14.07,x,g,o,f,e,NaN,w,17.80,...,s,y,w,u,w,t,g,NaN,d,w
3,p,14.17,f,h,e,f,e,NaN,w,15.77,...,s,y,w,u,w,t,p,NaN,d,w
4,p,14.64,x,h,o,f,e,NaN,w,16.53,...,s,y,w,u,w,t,p,NaN,d,w


In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("secondary_data.csv", sep=";")
# separar etiqueta y características
X = pd.get_dummies(df.drop(columns=["class"]), dtype=np.float32)
X = X.fillna(X.median(numeric_only=True)).fillna(0)
y = (df["class"] == "p").astype(np.int64).to_numpy()

# dividir manteniendo la proporción de cada clase
X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y, test_size=0.2, random_state=42, stratify=y
)

# escalar para facilitar el entrenamiento con SGD
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

D_in = X_train.shape[1]
D_out = len(np.unique(y_train))
print(f"Entradas: {D_in} | Clases: {D_out}")

Entradas: 119 | Clases: 2


In [16]:
# función de pérdida y derivada

def softmax(x):
    return torch.exp(x) / torch.exp(x).sum(axis=-1,keepdims=True)

def cross_entropy(output, target):
    logits = output[torch.arange(len(output)), target]
    loss = - logits + torch.log(torch.sum(torch.exp(output), axis=-1))
    loss = loss.mean()
    return loss

In [17]:
# X_train

In [18]:
torch.cuda.is_available()

False

In [19]:
print(X)

       cap-diameter  stem-height  stem-width  cap-shape_b  cap-shape_c  \
0             15.26        16.95       17.09          0.0          0.0   
1             16.60        17.99       18.19          0.0          0.0   
2             14.07        17.80       17.74          0.0          0.0   
3             14.17        15.77       15.98          0.0          0.0   
4             14.64        16.53       17.20          0.0          0.0   
...             ...          ...         ...          ...          ...   
61064          1.18         3.93        6.22          0.0          0.0   
61065          1.27         3.18        5.43          0.0          0.0   
61066          1.27         3.86        6.37          0.0          0.0   
61067          1.24         3.56        5.44          0.0          0.0   
61068          1.17         3.25        5.45          0.0          0.0   

       cap-shape_f  cap-shape_o  cap-shape_p  cap-shape_s  cap-shape_x  ...  \
0              0.0          0.0 

In [22]:
import torch

# 1. Definir dimensiones acordes al nuevo dataset
D_in = X_t.shape[1]   # 119
H = 64                # neuronas ocultas
D_out = 2             # 2 clases (comestible o venenoso)

# 2. Recrear el modelo con las dimensiones correctas
model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out)
)

In [23]:
epochs = 350
lr = 0.1
log_each = 20
l = []

for e in range(1, epochs + 1):
    # Forward
    y_pred = model(X_t)

    # Loss
    loss = cross_entropy(y_pred, Y_t)
    l.append(loss.item())

    # Reiniciar gradientes
    model.zero_grad()

    # Backpropagation
    loss.backward()

    # Actualización de pesos
    with torch.no_grad():
        for param in model.parameters():
            param -= lr * param.grad

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

Epoch 20/350 Loss 0.64893
Epoch 40/350 Loss 0.59932
Epoch 60/350 Loss 0.55078
Epoch 80/350 Loss 0.50729
Epoch 100/350 Loss 0.46958
Epoch 120/350 Loss 0.43682
Epoch 140/350 Loss 0.40808
Epoch 160/350 Loss 0.38262
Epoch 180/350 Loss 0.35989
Epoch 200/350 Loss 0.33949
Epoch 220/350 Loss 0.32110
Epoch 240/350 Loss 0.30444
Epoch 260/350 Loss 0.28930
Epoch 280/350 Loss 0.27552
Epoch 300/350 Loss 0.26292
Epoch 320/350 Loss 0.25137
Epoch 340/350 Loss 0.24076


Como puedes observar en el ejemplo, podemos calcular la salida del modelo con una simple línea. Luego calculamos la función de pérdida, y llamando a la función `backward` `Pytorch` se encarga de calcular las derivadas de la misma con respecto a todos los parámetros del modelo automáticamente (si no queremos acumular estos gradientes, nos aseguramos de llamar a la función `zero_grad` para ponerlos a cero antes de calcularlos). Por útlimo, podemos iterar por los parámetros del modelo aplicando la regla de actualización deseada (en este caso usamos `descenso por gradiente`).

In [14]:
from sklearn.metrics import accuracy_score

def evaluate(x):
    model.eval()
    device = next(model.parameters()).device
    x = x.to(device)
    y_pred = model(x)
    y_probas = softmax(y_pred)
    return torch.argmax(y_probas, axis=1)

y_pred = evaluate(torch.from_numpy(X_test).float())
accuracy_score(y_test, y_pred.cpu().numpy())

0.9613

Existen algunos tipos de capas que se comportan diferente en función de si estamos entrenando la red o usándola para generar predicciones. Podemos controlar el modo en el que queremos que esté nuestra red con las funciones `train` y `eval`.

## Optimizadores y Funciones de pérdida

En el ejemplo anterior hemos calculado la función de pérdida y aplicado la regla de optimización de forma manual. Sin embargo, `Pytorch` nos ofrece funcionalidad que nos abstrae estos cálculos ofreciendo además flexibilidad para aplicar diferentes funciones de pérdida o algoritmos de optimización de manera sencilla. Podemos encontrar diferentes funciones de pérdida ya implementadas en el paquete `torch.nn`.

In [24]:
criterion = torch.nn.CrossEntropyLoss()

Mientras que los optimizadores se encuentran en el paquete `torch.optim`

In [25]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.8)

Puedes ver la lista completa de funciones de pérdida y optimizadores disponibles en la [documentación](https://pytorch.org/docs/stable/index.html), aunque como ya has visto siempre puedes definir los tuyos propios fácilmente.

Una vez definidos estos dos objetos, nuestro bucle de entrenamiento se simplifica considerablemente.

In [27]:
import numpy as np
import torch
from sklearn.metrics import accuracy_score

# 1. Función de evaluación integrada
def evaluate(x):
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        y_pred = model(x)
        # argmax directo sobre los logits sin necesidad de softmax manual
        return torch.argmax(y_pred, dim=1)

# 2. Configurar dimensiones y modelo
D_in = X_train.shape[1]
H = 100
D_out = len(np.unique(y_train))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = torch.nn.Sequential(
    torch.nn.Linear(D_in, H),
    torch.nn.ReLU(),
    torch.nn.Linear(H, D_out),
).to(device)

# 3. Tensores
X_t = torch.from_numpy(X_train).float().to(device)
Y_t = torch.from_numpy(y_train).long().to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# 4. Bucle de entrenamiento
epochs = 100
log_each = 10
l = []
model.train()

for e in range(1, epochs + 1):
    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # backward y optimización
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

# 5. Evaluación final en conjunto de prueba
y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
acc = accuracy_score(y_test, y_pred.cpu().numpy())
print(f"\nExactitud final (Accuracy): {acc * 100:.2f}%")

Epoch 10/100 Loss 0.67043
Epoch 20/100 Loss 0.63002
Epoch 30/100 Loss 0.59610
Epoch 40/100 Loss 0.56528
Epoch 50/100 Loss 0.53705
Epoch 60/100 Loss 0.51130
Epoch 70/100 Loss 0.48786
Epoch 80/100 Loss 0.46650
Epoch 90/100 Loss 0.44697
Epoch 100/100 Loss 0.42903

Exactitud final (Accuracy): 91.89%


## Modelos custom

Si bien en muchos casos definir una `red neuronal` como una secuencia de capas es suficiente, en otros casos será un factor limitante. Un ejemplo son las redes residuales, en las que no sólo utilizamos la salida de una capa para alimentar la siguiente si no que, además, le sumamos su propia entrada. Este tipo de arquitectura no puede ser definida con la clase `Sequential`, y para ello necesitamos hacer un modelo *customizado*. Para ello, `Pytroch` nos ofrece la siguiente sintaxis.

In [28]:
# creamos una clase que hereda de `torch.nn.Module`

class ModeloPersonalizado(torch.nn.Module):

    # constructor
    def __init__(self, D_in, H, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloPersonalizado, self).__init__()

        # definimos nuestras capas
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

En primer lugar, necesitamos definir una nueva clase que herede de la clase `torch.nn.Module`. Esta clase madre aportará toda la funcionalidad esencial que necesita una `red neuronal` (soporte GPU, iterar por sus parámeteros, etc). Luego, en esta clase necesitamos definir mínimos dos funciones:

- `init`: en el constructor llamaremos al constructor de la clase madre y después definiremos todas las capas que querramos usar en la red.
- `forward`: en esta función definimos toda la lógica que aplicaremos desde que recibimos los inputs hasta que devolvemos los outputs.

En el ejemplo anterior simplemente hemos replicado la misma red (puedes conseguir el mismo efecto usando la clase `Sequential`).

In [29]:
model = ModeloPersonalizado(D_in, 100, D_out)
# Codigo para saber si el modelo esta votando los datos en las cantidades correctas
x_prueba = torch.randn(500, D_in)
print(x_prueba)
outputs = model(x_prueba)
outputs.shape

tensor([[ 0.5968, -1.0581,  1.2085,  ...,  1.1509, -0.0488,  2.2335],
        [-0.9665, -0.3810, -0.0890,  ..., -0.8090,  0.3085,  0.0320],
        [ 0.3710,  0.4152, -1.0850,  ..., -0.3360, -0.3077, -0.1190],
        ...,
        [ 0.2930,  0.7014,  0.5612,  ...,  1.2213,  0.9794,  1.5201],
        [-0.0907, -1.4452, -0.5789,  ..., -0.0779,  1.9283, -1.0280],
        [ 1.9560,  0.3285, -0.2025,  ..., -1.0536, -2.1650,  1.4936]])


torch.Size([500, 2])

Ahora, podemos entrenar nuestra red de la misma forma que lo hemos hecho anteriormente.

In [30]:
import numpy as np
from sklearn.metrics import accuracy_score

model = ModeloPersonalizado(D_in, 100, D_out)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
X_t = X_t.to(device)
Y_t = Y_t.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 0.66999
Epoch 20/100 Loss 0.63849
Epoch 30/100 Loss 0.60938
Epoch 40/100 Loss 0.58167
Epoch 50/100 Loss 0.55555
Epoch 60/100 Loss 0.53127
Epoch 70/100 Loss 0.50884
Epoch 80/100 Loss 0.48817
Epoch 90/100 Loss 0.46909
Epoch 100/100 Loss 0.45145


0.9079744555428197

Aquí puedes ver otro ejemplo de como definir un `MLP` con conexiones residuales, algo que no podemos hacer simplemente usando un modelo secuencial.

In [31]:
class ModelCustom2(torch.nn.Module):

    def __init__(self, D_in, H, D_out):
        super(ModelCustom2, self).__init__()
        self.fc1 = torch.nn.Linear(D_in, H)
        self.relu = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(H, D_out)

    def forward(self, x):
        x1 = self.fc1(x)
        x = self.relu(x1)
        x = self.fc2(x + x1)
        return x

In [32]:
import numpy as np
from sklearn.metrics import accuracy_score

model = ModelCustom2(D_in, 100, D_out).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100
log_each = 10
l = []
model.train()
for e in range(1, epochs + 1):

    # forward
    y_pred = model(X_t)

    # loss
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # ponemos a cero los gradientes
    optimizer.zero_grad()

    # Backprop (calculamos todos los gradientes automáticamente)
    loss.backward()

    # update de los pesos
    optimizer.step()

    if not e % log_each:
        print(f"Epoch {e}/{epochs} Loss {np.mean(l):.5f}")

y_pred = evaluate(torch.from_numpy(X_test).float().to(device))
accuracy_score(y_test, y_pred.cpu().numpy())

Epoch 10/100 Loss 0.57609
Epoch 20/100 Loss 0.50919
Epoch 30/100 Loss 0.46865
Epoch 40/100 Loss 0.44029
Epoch 50/100 Loss 0.41868
Epoch 60/100 Loss 0.40117
Epoch 70/100 Loss 0.38633
Epoch 80/100 Loss 0.37331
Epoch 90/100 Loss 0.36160
Epoch 100/100 Loss 0.35086


0.9065007368593417

De esta manera, tenemos mucha flexibilidad para definir nuestras redes.

## Accediendo a las capas de una red

En ocasiones queremos acceder a una capa en particular de nuestra red. Para ello, podemos acceder utilizando su nombre.

In [33]:
model

ModelCustom2(
  (fc1): Linear(in_features=119, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=2, bias=True)
)

In [34]:
model.fc1

Linear(in_features=119, out_features=100, bias=True)

También podemos acceder directamente a los tensores que contienen los parámetros con las propiedades adecuadas

In [35]:
model.fc1.weight

Parameter containing:
tensor([[-0.0406,  0.0548, -0.0738,  ...,  0.0448,  0.0438,  0.0697],
        [ 0.0114,  0.0582,  0.1053,  ..., -0.0169,  0.0831, -0.0049],
        [ 0.0377, -0.0178,  0.0321,  ..., -0.0128,  0.0133,  0.0207],
        ...,
        [-0.0188,  0.0042, -0.0160,  ...,  0.0910,  0.0497,  0.0769],
        [-0.0128, -0.0441,  0.0645,  ..., -0.0604,  0.0736, -0.0014],
        [-0.0030,  0.0478,  0.0602,  ...,  0.0063, -0.0223, -0.0771]],
       requires_grad=True)

In [36]:
model.fc1.bias

Parameter containing:
tensor([-0.0212, -0.0051,  0.0637, -0.0135, -0.0406, -0.0188, -0.0713,  0.0518,
         0.0067, -0.0448, -0.0852,  0.0423, -0.0122, -0.0688,  0.0373, -0.0476,
        -0.0539,  0.0491,  0.0627,  0.0370, -0.0755,  0.0505, -0.0459, -0.0794,
        -0.0196, -0.0317,  0.0170, -0.0015, -0.0024, -0.0032, -0.0248, -0.0585,
        -0.0097, -0.0409, -0.0453, -0.0582,  0.0640, -0.0047, -0.0467, -0.0442,
         0.0334,  0.0277,  0.0606, -0.0112, -0.0550,  0.0908,  0.0056, -0.0732,
        -0.0648,  0.0727, -0.0516,  0.0144,  0.0541, -0.0457, -0.0296, -0.0084,
        -0.0257, -0.0382, -0.0811, -0.0879,  0.0229,  0.0903,  0.0577,  0.0831,
         0.0559,  0.0099,  0.0641, -0.0783,  0.0317, -0.0400,  0.0378, -0.0079,
        -0.0706,  0.0026,  0.0581,  0.0170, -0.0056, -0.0094, -0.0140,  0.0876,
         0.0224, -0.0583,  0.0190,  0.0306, -0.0353, -0.0376,  0.0875,  0.0559,
        -0.0382, -0.0078,  0.0247, -0.0817, -0.0297,  0.0005, -0.0761,  0.0505,
        -0.0257,  

Es posible sobreescribir una capa de la siguiente manera

In [37]:
model.fc2 = torch.nn.Linear(100, 1)

model

ModelCustom2(
  (fc1): Linear(in_features=119, out_features=100, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=100, out_features=1, bias=True)
)

Ahora, la capa final de nuestra red tiene solo una salida. Esta nueva capa ha sido inicializada de manera aleatoria, por lo que esta nueva red no nos va a servir de mucho. Sin embargo, podríamos volver a entrenar esta red en otro problema en el que solo necesitemos una salida aprovechando los pesos que ya hemos entrenado anteriormente con el dataset MNIST. Esto es la base del *transfer learning*, una técnica que utilizaremos muchísimo más adelante y la cual explicaremos en detalle.

A continuación encontrarás varios trucos a la hora de crear redes neuronales a partir de otras que te pueden resultar útiles.

In [38]:
# obtener una lista con las capas de una red

list(model.children())

[Linear(in_features=119, out_features=100, bias=True),
 ReLU(),
 Linear(in_features=100, out_features=1, bias=True)]

In [39]:
# crear nueva red a partir de la lista (excluyendo las útlimas dos capa)

new_model = torch.nn.Sequential(*list(model.children())[:-2])
new_model

Sequential(
  (0): Linear(in_features=119, out_features=100, bias=True)
)

In [40]:
# crear nueva red a partir de la lista (excluyendo las útlima capa)

new_model = torch.nn.ModuleList(list(model.children())[:-1])
new_model

ModuleList(
  (0): Linear(in_features=119, out_features=100, bias=True)
  (1): ReLU()
)

## Resumen

En este post hemos visto la funcionalidad que `Pytorch` nos ofrece a la hora de definir y entrenar nuestras `redes neuronales`. El paquete `torch.nn` contiene todo lo necesario para diseñar nuestros modelos, ya sea de manera secuencial o con una clase *custom* para arquitecturas más complicadas. También nos da muchas funciones de pérdida que podemos usar directamente para entrenar las redes. Te recomiendo encarecidamente que le eches un vistazo a la [documentación](https://pytorch.org/docs/stable/nn.html) par hacerte una idea de todo lo que puedes hacer. También hemos visto como el paquete `torch.optim` nos oferece algoritmos de optimización que también nos hacen la vida más fácil a la hora de entrenar nuestras redes.